In [80]:
import pandas as pd
import glob
import os

RAW_DIR = '../../data/norming_study/raw/d_pilot_prolific/'

In [81]:
# load all half csvs (skip demographics)
half_files = glob.glob(os.path.join(RAW_DIR, '*_half.csv'))
raw = pd.concat([pd.read_csv(f) for f in half_files], ignore_index=True)

print(f'{len(half_files)} files loaded, {len(raw)} total rows')
raw.head()

4 files loaded, 62 total rows


,trial_type,subjectID,prolificID,studyID,sessionID,captchaOk,DEBUG,axisOrder,colorAssignment,vizMode,...,not_able_willing_count,not_able_not_willing_count,total,abilityResponse,willingnessResponse,firstInteractionRT,stage2RT,stage3RT,trialRT,suspicious
0,whyask-grid-trial,14ouazphvd,5e3d71c28eb5270274d528b8,6a0f47a92147475d2b7d91ea,6a0fffaf1547a7c0b31e9082,err,0,AW,"{""AW"":""#1F5572"",""NAW"":""#D69A57"",""ANW"":""#5C8FA8...",figures,...,18,72,100,10,24,18395,20543,22924,50318,False
1,whyask-grid-trial,14ouazphvd,5e3d71c28eb5270274d528b8,6a0f47a92147475d2b7d91ea,6a0fffaf1547a7c0b31e9082,err,0,AW,"{""AW"":""#1F5572"",""NAW"":""#D69A57"",""ANW"":""#5C8FA8...",figures,...,32,48,100,20,40,9834,11877,16055,22879,False
2,whyask-grid-trial,14ouazphvd,5e3d71c28eb5270274d528b8,6a0f47a92147475d2b7d91ea,6a0fffaf1547a7c0b31e9082,err,0,AW,"{""AW"":""#1F5572"",""NAW"":""#D69A57"",""ANW"":""#5C8FA8...",figures,...,60,0,100,40,100,14597,16624,19235,28817,False
3,whyask-grid-trial,14ouazphvd,5e3d71c28eb5270274d528b8,6a0f47a92147475d2b7d91ea,6a0fffaf1547a7c0b31e9082,err,0,AW,"{""AW"":""#1F5572"",""NAW"":""#D69A57"",""ANW"":""#5C8FA8...",figures,...,1,9,100,90,10,9311,11130,15546,46523,False
4,whyask-grid-trial,14ouazphvd,5e3d71c28eb5270274d528b8,6a0f47a92147475d2b7d91ea,6a0fffaf1547a7c0b31e9082,err,0,AW,"{""AW"":""#1F5572"",""NAW"":""#D69A57"",""ANW"":""#5C8FA8...",figures,...,21,9,100,70,49,15411,17618,20972,25589,False


In [82]:
# split trials (attn checks removed from study)
trials = raw[raw['half'].isin([1, 2])].copy()
# attn = raw[raw['half'] == 'attn'].copy()

print(f'trials: {len(trials)} rows, {trials["subjectID"].nunique()} subjects')
print(f'subjects: {sorted(trials["subjectID"].unique())}')

trials: 62 rows, 2 subjects
subjects: ['14ouazphvd', '2fnq4uzyh7']


In [83]:
# load demographics
demo_files = glob.glob(os.path.join(RAW_DIR, '*_demographics.csv'))
demo = pd.concat([pd.read_csv(f) for f in demo_files], ignore_index=True)

print(f'{len(demo)} participants\n')
print(demo[['subjectID', 'age', 'gender', 'race', 'education']].to_string(index=False))

2 participants

 subjectID  age gender  race    education
14ouazphvd   73 Female Other   Bachelor's
2fnq4uzyh7   40   Male White Some college


In [84]:
# timing + visibility changes
print(demo[['subjectID', 'totalDurationMs', 'visibilityChanges']]
      .assign(duration_min=lambda df: (df['totalDurationMs'] / 60000).round(1))
      [['subjectID', 'duration_min', 'visibilityChanges']]
      .to_string(index=False))

 subjectID  duration_min  visibilityChanges
14ouazphvd          19.9                  0
2fnq4uzyh7          13.0                  0


In [85]:
# captcha — exclude anyone who didn't pass server verification
trials_clean = trials.copy()
demo_clean   = demo.copy()

failed = demo_clean[demo_clean['captchaOk'] != 1][['subjectID', 'prolificID', 'captchaOk']]
if len(failed):
    print(f'{len(failed)} subject(s) failed captcha:')
    print(failed.to_string(index=False))
    trials_clean = trials_clean[~trials_clean['subjectID'].isin(failed['subjectID'])]
    demo_clean   = demo_clean[~demo_clean['subjectID'].isin(failed['subjectID'])]
    print(f'\nexcluded → {trials_clean["subjectID"].nunique()} subjects remain')
else:
    print('all passed captcha')

2 subject(s) failed captcha:
 subjectID               prolificID captchaOk
14ouazphvd 5e3d71c28eb5270274d528b8       err
2fnq4uzyh7 60ddc62b071a009e0b0c2922       err

excluded → 0 subjects remain


In [86]:
# strategy, technical issues, other feedback (first 10)
for _, row in demo.head(20)[['subjectID', 'strategy', 'technicalIssues', 'feedback']].iterrows():
    print(f"{row['subjectID']}:")
    print(f"  strategy: {row['strategy']}")
    print(f"  issues:   {row['technicalIssues']}")
    print(f"  feedback: {row['feedback']}")
    print()

14ouazphvd:
  strategy: I imagined what I would do in each case and also thought about the likelihood that many people could do a certain thing. For example almost everyone would be willing, if able to look after a child if there is an emergency, but not everyone is able to play a cello solo, even if they would like to.
  issues:   nan
  feedback: Just right

2fnq4uzyh7:
  strategy: I thought about my close group of friends and as I know them well I could tell what they would be able or not be able to do.
  issues:   nan
  feedback: about right



In [87]:
# quick look at trial responses
common_cols = ['subjectID', 'trial_type', 'axisOrder', 'itemID', 'actionPhrase',
               'abilityResponse', 'willingnessResponse', 'trialRT', 'suspicious']
print(trials[common_cols].to_string())
print()

# joint-specific
grid_df = trials[trials['trial_type'] == 'whyask-grid-trial']
if len(grid_df):
    print('\njoint (sx / syL / syR / quadrant counts):')
    jcols = ['subjectID', 'itemID', 'sx', 'syL', 'syR',
             'able_willing_count', 'able_not_willing_count',
             'not_able_willing_count', 'not_able_not_willing_count']
    print(grid_df[[c for c in jcols if c in grid_df.columns]].to_string())

avg_rt = trials['trialRT'].mean() / 1000
print(f'\navg trial RT: {avg_rt:.1f}s')

     subjectID         trial_type axisOrder    itemID                                                                        actionPhrase  abilityResponse  willingnessResponse  trialRT  suspicious
0   14ouazphvd  whyask-grid-trial        AW        48                              use this to get us back to shore before the storm hits               10                   24    50318       False
1   14ouazphvd  whyask-grid-trial        AW        98                                                            put it out of its misery               20                   40    22879       False
2   14ouazphvd  whyask-grid-trial        AW        38                        hot-wire this car so we can get them to the hospital in time               40                  100    28817       False
3   14ouazphvd  whyask-grid-trial        AW        83                                         send this to the team and say it's from you               90                   10    46523       False
4   14ouazphvd 

In [88]:
# save processed
PROC_DIR = '../../data/norming_study/processed/c_pilot_prolific/'
os.makedirs(PROC_DIR, exist_ok=True)

# drop trial-level cols from demo
trial_cols = ['trialIndex', 'itemID', 'actionPhrase', 'abilityResponse', 'abilityRT',
              'abilityDragged', 'willingnessResponse', 'willingnessRT', 'willingnessDragged',
              'trialRT', 'suspicious', 'targetValue', 'passed']
demo_out = demo_clean.drop(columns=[c for c in trial_cols if c in demo_clean.columns])

trials_clean.to_csv(os.path.join(PROC_DIR, 'trials.csv'), index=False)
demo_out.to_csv(os.path.join(PROC_DIR, 'demographics.csv'), index=False)

print(f'saved trials.csv ({len(trials_clean)} rows) and demographics.csv ({len(demo_out)} rows)')
print(f'demo cols: {list(demo_out.columns)}')

saved trials.csv (0 rows) and demographics.csv (0 rows)
demo cols: ['trial_type', 'subjectID', 'prolificID', 'studyID', 'sessionID', 'captchaOk', 'DEBUG', 'axisOrder', 'colorAssignment', 'vizMode', 'half', 'age', 'gender', 'race', 'education', 'strategy', 'technicalIssues', 'feedback', 'visibilityChanges', 'totalDurationMs']
